# 自定义input()输入函数
1. 现以实现封装成包，自动导入等，使用时直接按照下面格式输入：
    ```python
    input = make_input(
        """
        123
        456
        789
        """
    )

    for i in range(3):
        print(input) # 代码会自动按行读取三引号里的东西
    ```

In [5]:
# 定义一个模拟输入函数
def make_input(inputs: str):
    lines = iter(inputs.strip().splitlines())
    def fake_input(prompt=""):
        try:
            return next(lines)
        except StopIteration:
            raise EOFError("没有更多的输入啦")
    return fake_input


# 用法
input = make_input(
"""
123 987 654
456
789
""")

a = list(map(int, input().split()))
b = int(input())
c = int(input())
print(a, b, c)  # 输出: [123, 987, 654] 456 789

[123, 987, 654] 456 789


# 测试pytorch_GPU加速

## 检查GPU是否可用

In [5]:
import torch

if torch.cuda.is_available():
    print(f"GPU 可用，使用设备: {torch.cuda.get_device_name(0)}")
else:
    print("GPU 不可用，使用 CPU")

GPU 可用，使用设备: NVIDIA GeForce RTX 3060 Laptop GPU


## 准备数据

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

100.0%
100.0%
100.0%
100.0%


## 定义模型并迁移到GPU

In [7]:
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleNN().to(device)

## 模型训练

In [8]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.4f}')

for epoch in range(1, 6): # 训练5个epoch
    train(model, device, train_loader, optimizer, epoch)

Epoch: 1 [0/60000] Loss: 2.3637
Epoch: 1 [6400/60000] Loss: 0.4728
Epoch: 1 [12800/60000] Loss: 0.5163
Epoch: 1 [19200/60000] Loss: 0.3016
Epoch: 1 [25600/60000] Loss: 0.2777
Epoch: 1 [32000/60000] Loss: 0.3042
Epoch: 1 [38400/60000] Loss: 0.3821
Epoch: 1 [44800/60000] Loss: 0.2212
Epoch: 1 [51200/60000] Loss: 0.1255
Epoch: 1 [57600/60000] Loss: 0.2690
Epoch: 2 [0/60000] Loss: 0.2272
Epoch: 2 [6400/60000] Loss: 0.2135
Epoch: 2 [12800/60000] Loss: 0.1700
Epoch: 2 [19200/60000] Loss: 0.3212
Epoch: 2 [25600/60000] Loss: 0.3946
Epoch: 2 [32000/60000] Loss: 0.1287
Epoch: 2 [38400/60000] Loss: 0.2261
Epoch: 2 [44800/60000] Loss: 0.0980
Epoch: 2 [51200/60000] Loss: 0.1588
Epoch: 2 [57600/60000] Loss: 0.1899
Epoch: 3 [0/60000] Loss: 0.1614
Epoch: 3 [6400/60000] Loss: 0.1221
Epoch: 3 [12800/60000] Loss: 0.0719
Epoch: 3 [19200/60000] Loss: 0.2390
Epoch: 3 [25600/60000] Loss: 0.1737
Epoch: 3 [32000/60000] Loss: 0.3315
Epoch: 3 [38400/60000] Loss: 0.0278
Epoch: 3 [44800/60000] Loss: 0.0814
Epoch: 